<a href="https://colab.research.google.com/github/utamiu1807/-utami-creditcardclustering/blob/main/ML_Foundations_Classification_Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
import xgboost as xgb
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, ConfusionMatrixDisplay
from google.colab import drive
from google.colab import files

## Read Dataset

In [ ]:
data = pd.read_csv('https://www.dropbox.com/scl/fi/lle2wyziwywf8kxaq56na/telco_churn.csv?rlkey=kh9571prkkg8uki7mahnwunml&st=da4ou0gy&dl=1')

data.head()

## Data Exploration and Pre-processing

In [ ]:
# Drop unnecessary columns
data = data.drop(['Country', 'State', 'ChurnReason', 'ChurnLabel', "CustomerID"], axis=1)

In [ ]:
data.shape

In [ ]:
# Show all columns that have missing values
data.columns[data.isnull().any()]

In [ ]:
# Drop rows with missing values
data = data.dropna()
data.shape

In [ ]:
# Starter predictor variables
selected_columns = [
    "TenureMonths",
    "Contract",
    "MonthlyCharges",
    "InternetService",
    "TechSupport",
    "ChurnValue"
]

data = data[selected_columns]

In [ ]:
# Your exploration here

## Split Train and Test

In [ ]:
# Convert categorical features to numerical using one-hot encoding
data = pd.get_dummies(data, drop_first=True)

# Define features (X) and target (y)
X = data.drop('ChurnValue', axis=1)
y = data['ChurnValue']

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

## Logistic Regression

In [ ]:
# Initialize the Logistic Regression model
model = LogisticRegression()

# Train the model
model.fit(X_train, y_train)

# Make predictions on the test set
y_pred = model.predict(X_test)

# Evaluate the model
print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

cm = confusion_matrix(y_test, y_pred)

# Create the confusion matrix display
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Retained', 'Churned'])

# Plot the confusion matrix
disp.plot(cmap='Blues')
plt.title('Logistic Regression Confusion Matrix')
plt.show()

In [ ]:
feature_names = data.drop('ChurnValue', axis=1).columns

coefficients = model.coef_[0]

coefficients_df = pd.DataFrame({'Feature': feature_names, 'Coefficient': coefficients})
coefficients_df

## K Nearest Neighbors

In [ ]:
# Initialize the KNN model
knn_model = KNeighborsClassifier(n_neighbors=20)  # You can adjust n_neighbors

# Train the model
knn_model.fit(X_train, y_train)

# Make predictions on the test set
knn_y_pred = knn_model.predict(X_test)

# Evaluate the model
print("KNN Accuracy:", accuracy_score(y_test, knn_y_pred))
print("\nKNN Classification Report:\n", classification_report(y_test, knn_y_pred))

cm = confusion_matrix(y_test, knn_y_pred)

# Create the confusion matrix display
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Retained', 'Churned'])

# Plot the confusion matrix
disp.plot(cmap='Blues')
plt.title('KNN Confusion Matrix')
plt.show()

## Random Forest

In [ ]:
# Initialize the Random Forest model
rf_model = RandomForestClassifier(random_state=42)

# Train the model
rf_model.fit(X_train, y_train)

# Make predictions on the test set
rf_y_pred = rf_model.predict(X_test)

# Evaluate the model
print("Random Forest Accuracy:", accuracy_score(y_test, rf_y_pred))
print("\nRandom Forest Classification Report:\n", classification_report(y_test, rf_y_pred))

cm = confusion_matrix(y_test, rf_y_pred)

# Create the confusion matrix display
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Retained', 'Churned'])

# Plot the confusion matrix
disp.plot(cmap='Blues')
plt.title('Random Forest Confusion Matrix')
plt.show()

## Make Predictions on Unknown Data

In [ ]:
prediction_data = pd.read_csv('https://www.dropbox.com/scl/fi/o4g2rytrm9off6rroh233/telco_churn_test_data.csv?rlkey=xpe5qjslooe1y79p1esjd7ps2&st=lots4ru3&dl=1')

prediction_data.head()

In [ ]:
# use the same pre-processing on this new dataset that we did on the original training dataset

prediction_data = prediction_data.drop(['Country', 'State', 'ChurnReason', 'ChurnLabel', "CustomerID"], axis=1)

prediction_data = prediction_data[selected_columns]

prediction_data = pd.get_dummies(prediction_data, drop_first=True)

X_pred = prediction_data.drop('ChurnValue', axis=1)

# make sure we have the same columns in this dataset as in training
training_features = X.columns
X_pred = X_pred.reindex(columns=training_features, fill_value=0)

scaler = StandardScaler()
X_pred = scaler.fit_transform(X_pred)


In [ ]:
# Predict outcomes on new dataset

# Add the predictions to the prediction_data DataFrame
prediction_data['ChurnValue'] = knn_model.predict(X_pred)
prediction_data['Churn_Probability'] = knn_model.predict_proba(X_pred)[:, 1]

prediction_data.head()

In [ ]:
# if the column Churn_Probability is in the top 30% of the dataframe assign it to a variable "ChurnCategory"

churn_threshold = prediction_data['Churn_Probability'].quantile(0.7)
prediction_data.loc[prediction_data['Churn_Probability'] >= churn_threshold, "ChurnCategory"] = "High"

In [ ]:
# Save files for adding to Google Sheets

prediction_data.to_csv('team_predictions.csv', index=False)
files.download('team_predictions.csv')

### ML Foundations Spring 2026

[Post your ChurnValue Predictions to Google Sheets](https://docs.google.com/spreadsheets/d/11-usI5vKD-VmTgcY2jYElBHxJtO_KLp411tuYb79vh4/edit?usp=sharing)